# Pareja 2 — Partes D, E y F: MM/GBSA y descomposición por residuo

Notebook adaptado para **System_2A** (WT/inicial, $K_D = 23\text{ nM}$) y **System_2B** (madurada, $K_D = 0.15\text{ nM}$), $\Delta\Delta G_{\text{exp}} \approx -2.98\text{ kcal/mol}$.

**No repite la dinámica molecular.** Usa directamente las trayectorias ya corregidas por PBC (`proteina.tpr`, `proteina.xtc`, `proteina.ndx`) de `B_dinamica/2A_colab/` y `B_dinamica/2B_colab/` empaquetadas en `datos_md_pareja2.zip` (~28 MB).

**Cómo ejecutarlo en Google Colab:**
1. Ejecuta la celda 1 (instala `condacolab` y reinicia el kernel; es normal).
2. Ejecuta la celda 2 (instala GROMACS y `gmx_MMPBSA` en entorno Python 3.12).
3. Ejecuta la celda 3 y sube `D_F_energia/datos_md_pareja2.zip`.
4. Ejecuta las celdas 4 a 8 en orden y descarga `resultados_energia_pareja2.zip`.


## 0. Entorno (GROMACS + gmx_MMPBSA)


In [ ]:
# 1. Conda en Colab (reinicia el kernel automáticamente; luego sigue con la celda 2)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()


In [ ]:
# 2. GROMACS (CPU, suficiente para gmx_MMPBSA) y entorno separado para gmx_MMPBSA (exige Python <3.13)
!mamba install -y -c conda-forge -c bioconda --no-pin gromacs
!ln -sf /usr/local/bin/gmx /usr/bin/gmx
!conda create -y -q -n gmxMMPBSA -c conda-forge python=3.12 "ambertools>=24.8,<27" "mpi4py>=4.0.1,<5" "numpy>=1.26.4,<2" "matplotlib>=3.8,<4" "scipy>=1.14.1,<2" "pandas>=2.2,<3" "seaborn>=0.13,<0.14" pip
!/usr/local/envs/gmxMMPBSA/bin/python -m pip install -q gmx_MMPBSA==1.7.1
!gmx --version | grep -E "GROMACS version"
!PATH=/usr/local/envs/gmxMMPBSA/bin:$PATH AMBERHOME=/usr/local/envs/gmxMMPBSA gmx_MMPBSA --version


## 1. Subir `datos_md_pareja2.zip` y preparar topología seca + `ref.pdb`


In [ ]:
# 3. Subir datos_md_pareja2.zip (generado desde B_dinamica/2A_colab y 2B_colab)
import os, shutil
from google.colab import files

WORK = "/content/work"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "{zip_name}" -d {WORK}
!ls -lh {WORK}/2A_colab {WORK}/2B_colab


In [ ]:
%%bash
# 4. Crear topol_dry.top (sin SOL ni CL para que coincida con proteina.tpr/proteina.xtc)
#    y generar ref.pdb con cadenas explícitas: A = RNasa A (Protein_chain1), B = Nanobody (Protein_chain2)
set -euo pipefail
WORK=/content/work

for SIS in 2A_colab 2B_colab; do
  cd "$WORK/$SIS"
  grep -vE '^(SOL|CL|NA)[[:space:]]+' topol.top > topol_dry.top
  echo "0" | gmx trjconv -s proteina.tpr -f proteina.xtc -dump 0 -o ref_raw.pdb >/dev/null 2>&1

  python3 - <<'PY'
n_chain1 = 0
in_c1 = False
for line in open("proteina.ndx"):
    s = line.strip()
    if s.startswith("["):
        in_c1 = (s == "[ Protein_chain1 ]")
    elif in_c1 and s:
        n_chain1 += len(s.split())

out = []
atom_idx = 0
for line in open("ref_raw.pdb"):
    if line.startswith(("ATOM  ", "HETATM")):
        atom_idx += 1
        chain = "A" if atom_idx <= n_chain1 else "B"
        line = line[:21] + chain + line[22:]
    out.append(line)
open("ref.pdb", "w").write("".join(out))
print(f"ref.pdb listo: {atom_idx} átomos (Cadena A={n_chain1}, Cadena B={atom_idx - n_chain1})")
PY
done


## 2. D–F. MM/GBSA y descomposición por residuo (`igb=8`, 101 snapshots)

Mismo protocolo para **2A** y **2B**:
- **Receptor:** `Protein_chain2` (Nanobody, cadena B) · **Ligando:** `Protein_chain1` (RNasa A, cadena A)
- **Ventana:** últimos 500 ps (`startframe=501`, `endframe=1001`, `interval=5` → 101 snapshots)
- **Modelo GB:** GB-Neck2 (`igb=8`, radios `mbondi3` / `PBRadii=4`), `saltcon=0.150` M, `temperature=298.15` K
- **Descomposición:** `idecomp=2, dec_verbose=3, csv_format=1, print_res="all"` (todos los residuos para comparar mutados, vecinos, interfaz y antígeno sin perder las mutaciones indirectas).


In [ ]:
%%bash
# 5. Escribir archivo de entrada y ejecutar gmx_MMPBSA para 2A y 2B en paralelo
set -euo pipefail
WORK=/content/work
ENVPY=/usr/local/envs/gmxMMPBSA
export PATH=$ENVPY/bin:$PATH
export AMBERHOME=$ENVPY

run_mmpbsa() {
  SIS=$1
  cd "$WORK/$SIS"
  mkdir -p mmgbsa
  cd mmgbsa
  cp ../topol_dry.top ../topol_Protein_chain_*.itp ../posre_Protein_chain_*.itp .

  cat <<EOT > mmgbsa.in
&general
  sys_name="$SIS", startframe=501, endframe=1001, interval=5,
  temperature=298.15, PBRadii=4, keep_files=0, gmx_path="/usr/local/bin",
/
&gb
  igb=8, saltcon=0.150,
/
&decomp
  idecomp=2, dec_verbose=3, csv_format=1, print_res="all",
/
EOT

  gmx_MMPBSA -O -i mmgbsa.in -cs ../proteina.tpr -ci ../proteina.ndx     -cg Protein_chain2 Protein_chain1 -ct ../proteina.xtc     -cp topol_dry.top -cr ../ref.pdb     -o FINAL_RESULTS_MMPBSA.dat -eo FINAL_RESULTS_MMPBSA.csv     -do FINAL_DECOMP_MMPBSA.dat -deo FINAL_DECOMP_MMPBSA.csv     -nogui > gmx_MMPBSA.log 2>&1 || { tail -40 gmx_MMPBSA.log; exit 1; }
  echo "$SIS: gmx_MMPBSA terminado"
}

run_mmpbsa 2A_colab & p1=$!
run_mmpbsa 2B_colab & p2=$!
e=0; wait $p1 || e=1; wait $p2 || e=1
exit $e


## 3. Análisis de resultados (componentes, compensación, descomposición y comparación experimental)


In [ ]:
%%writefile /content/work/analisis_pareja2.py
import csv, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORK = "/content/work"
OUT = f"{WORK}/resultados_energia"
os.makedirs(OUT, exist_ok=True)
DDG_EXP = -2.98  # kcal/mol (KD: 23 nM -> 0.15 nM a 298.15 K)

plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})

def bloques(ruta):
    salida, titulo, cab, filas = [], "", None, []
    def cerrar():
        nonlocal cab, filas
        if cab and filas:
            salida.append((titulo, pd.DataFrame(filas, columns=cab).drop(columns="Frame #").astype(float)))
        cab, filas = None, []
    with open(ruta, encoding="utf-8") as fh:
        for row in csv.reader(fh):
            if not "".join(row).strip():
                cerrar()
            elif row[0] == "Frame #":
                cab, filas = row, []
            elif cab and row[0].isdigit():
                filas.append(row)
            else:
                cerrar()
                titulo = row[0].strip()
    cerrar()
    return salida

def delta(ruta):
    return [d for t, d in bloques(ruta) if t == "Delta Energy Terms"][0]

def decomp(ruta):
    cols = ["Internal", "vdW", "Ele", "Polar", "NoPolar", "TOTAL"]
    datos, seccion, token = {}, None, None
    with open(ruta, encoding="utf-8") as fh:
        for row in csv.reader(fh):
            c = row[0].strip() if row else ""
            if c in ("Complex:", "Receptor:", "Ligand:", "DELTAS:"):
                seccion, token = c, None
            elif c.endswith("Decomposition Contribution (TDC)"):
                token = "TDC"
            elif c.endswith(("(SDC)", "(BDC)")):
                token = c[-4:-1]
            elif seccion == "DELTAS:" and token and c.isdigit() and len(row) == 8:
                # Formato "R::TYR:101" (sin cadena): R = receptor = nanobody (B), L = ligando = RNasa (A)
                p = row[1].split(":")
                r = datos.setdefault(({"R": "B", "L": "A"}[p[0]], int(p[3])), {"res": p[2]})
                for k, v in zip(cols, row[2:]):
                    r.setdefault(f"{token}_{k}", []).append(float(v))
    return datos

def block_sem(x, n=5):
    medias = [b.mean() for b in np.array_split(np.asarray(x, float), n)]
    return float(np.std(medias, ddof=1) / np.sqrt(n))

# --- D y E: Componentes energéticos globales
d2a = delta(f"{WORK}/2A_colab/mmgbsa/FINAL_RESULTS_MMPBSA.csv")
d2b = delta(f"{WORK}/2B_colab/mmgbsa/FINAL_RESULTS_MMPBSA.csv")

terminos = {
    "VDWAALS": "ΔE vdW",
    "EEL": "ΔE electrostática",
    "EGB": "ΔG solvatación polar (GB)",
    "ESURF": "ΔG solvatación no polar",
    "GGAS": "ΔG gas (vdW + EEL)",
    "GSOLV": "ΔG solv (GB + SURF)",
    "TOTAL": "ΔG total unión",
}
filas = []
for k, nombre in terminos.items():
    w, m = d2a[k], d2b[k]
    ew, em = block_sem(w), block_sem(m)
    filas.append(dict(
        termino=nombre,
        dG_2A=w.mean(), sem_2A=ew,
        dG_2B=m.mean(), sem_2B=em,
        ddG_2B_menos_2A=m.mean() - w.mean(),
        ddG_sem=float(np.hypot(ew, em)),
    ))
g = pd.DataFrame(filas)
g.round(2).to_csv(f"{OUT}/mmgbsa_componentes.csv", index=False)

print("=== D y E. COMPONENTES ENERGÉTICOS (kcal/mol) ===")
print(g.round(2).to_string(index=False))

tot = g[g.termino == "ΔG total unión"].iloc[0]
eel = g[g.termino == "ΔE electrostática"].iloc[0]
egb = g[g.termino == "ΔG solvatación polar (GB)"].iloc[0]
print(f"\nΔΔG_calc (2B - 2A) = {tot.ddG_2B_menos_2A:+.2f} ± {tot.ddG_sem:.2f} kcal/mol | ΔΔG_exp = {DDG_EXP:+.2f} kcal/mol")
print(f"Compensación electrostática neta ΔΔ(EEL + EGB) = {eel.ddG_2B_menos_2A + egb.ddG_2B_menos_2A:+.2f} kcal/mol")

fig, ax = plt.subplots(figsize=(7.5, 3.6), layout="constrained")
p = g[g.termino.isin(["ΔE vdW", "ΔE electrostática", "ΔG solvatación polar (GB)", "ΔG solvatación no polar", "ΔG total unión"])][::-1]
colores = ["#2a9d8f" if v < 0 else "#e76f51" for v in p.ddG_2B_menos_2A]
ax.barh(p.termino, p.ddG_2B_menos_2A, xerr=p.ddG_sem, color=colores, height=0.55, error_kw=dict(lw=1))
ax.axvline(0, color="#444", lw=1)
ax.axvline(DDG_EXP, color="#1d3557", ls="--", lw=1.2, label=f"ΔΔG exp = {DDG_EXP} kcal/mol")
ax.legend(frameon=False)
ax.set(xlabel="ΔΔG = 2B − 2A (kcal/mol)", title="Cambio energético por componente (Pareja 2)")
fig.savefig(f"{OUT}/mmgbsa_componentes.png")
plt.close(fig)

# Serie temporal de ΔG_total (500-1000 ps) para verificar estabilidad y ausencia de outliers
tiempo_ps = np.linspace(500, 1000, len(d2a))
fig, ax = plt.subplots(figsize=(7.5, 3.2), layout="constrained")
for df_s, lbl, col in ((d2a, "System_2A (inicial)", "#6c757d"), (d2b, "System_2B (madurada)", "#2a9d8f")):
    y = df_s["TOTAL"].to_numpy()
    ax.plot(tiempo_ps, y, alpha=0.35, lw=1, color=col)
    mm = pd.Series(y).rolling(10, min_periods=1, center=True).mean()
    ax.plot(tiempo_ps, mm, lw=2, color=col, label=f"{lbl} (media={y.mean():.1f})")
ax.set(xlabel="Tiempo (ps)", ylabel="ΔG unión (kcal/mol)", title="Estabilidad de ΔG_total en la ventana de análisis (500–1000 ps)")
ax.legend(frameon=False)
fig.savefig(f"{OUT}/mmgbsa_serie_tiempo.png")
plt.close(fig)

# --- F: Descomposición por residuo (TDC, SDC = cadena lateral, BDC = esqueleto)
r2a = decomp(f"{WORK}/2A_colab/mmgbsa/FINAL_DECOMP_MMPBSA.csv")
r2b = decomp(f"{WORK}/2B_colab/mmgbsa/FINAL_DECOMP_MMPBSA.csv")
comunes = sorted(set(r2a.keys()) & set(r2b.keys()))
solo_2a = sorted(set(r2a.keys()) - set(r2b.keys()))
solo_2b = sorted(set(r2b.keys()) - set(r2a.keys()))
print(f"\nResiduos comunes: {len(comunes)} | Solo en 2A: {solo_2a} | Solo en 2B: {solo_2b}")

# Nota: este análisis está replicado (y ampliado) en scripts/analisis_energia.py, que corre sin Colab.
MUTADOS_B = {28, 29, 101, 102, 104, 105, 106, 107, 110}
INTERFAZ_NO_MUT_B = {27, 30, 31, 32, 33, 99, 100, 108, 109}
CLAVE_ANT_A = {60, 62, 70, 73, 111, 112, 115}

filas_r = []
for k in comunes:
    w, m = r2a[k], r2b[k]
    dg_w, dg_m = float(np.mean(w["TDC_TOTAL"])), float(np.mean(m["TDC_TOTAL"]))
    cat = "Otro"
    if k[0] == "B" and k[1] in MUTADOS_B:
        cat = "Mutado (NB)"
    elif k[0] == "B" and k[1] in INTERFAZ_NO_MUT_B:
        cat = "Interfaz no mutado (NB)"
    elif k[0] == "A" and k[1] in CLAVE_ANT_A:
        cat = "Antígeno clave (RNasa)"
    fila = dict(
        cadena=k[0], num=k[1], res_2A=w["res"], res_2B=m["res"], categoria=cat,
        dG_2A=dg_w, dG_2B=dg_m, ddG_i=dg_m - dg_w,
        ddG_sem=float(np.hypot(block_sem(w["TDC_TOTAL"]), block_sem(m["TDC_TOTAL"]))),
        dd_vdW=float(np.mean(m["TDC_vdW"]) - np.mean(w["TDC_vdW"])),
        dd_Ele=float(np.mean(m["TDC_Ele"]) - np.mean(w["TDC_Ele"])),
        dd_Polar=float(np.mean(m["TDC_Polar"]) - np.mean(w["TDC_Polar"])),
        dd_NoPolar=float(np.mean(m["TDC_NoPolar"]) - np.mean(w["TDC_NoPolar"])),
    )
    for tok, nombre in (("SDC", "SDC"), ("BDC", "BDC")):
        if f"{tok}_TOTAL" in w and f"{tok}_TOTAL" in m:
            s_w, s_m = float(np.mean(w[f"{tok}_TOTAL"])), float(np.mean(m[f"{tok}_TOTAL"]))
            fila[f"{nombre}_2A"] = s_w
            fila[f"{nombre}_2B"] = s_m
            fila[f"dd_{nombre}"] = s_m - s_w
    filas_r.append(fila)

df_r = pd.DataFrame(filas_r).sort_values("ddG_i", key=abs, ascending=False)
df_r.round(3).to_csv(f"{OUT}/descomposicion_ddg_completa.csv", index=False)

cols_tab = ["cadena", "num", "res_2A", "res_2B", "dG_2A", "dG_2B", "ddG_i", "dd_SDC", "dd_BDC", "dd_vdW", "dd_Ele", "dd_Polar"]
cols_tab = [c for c in cols_tab if c in df_r.columns]

print("\n=== F. TOP 20 RESIDUOS QUE MÁS CAMBIAN (|ΔΔG_i|) ===")
print(df_r.head(20)[["cadena", "num", "res_2A", "res_2B", "categoria", "dG_2A", "dG_2B", "ddG_i", "ddG_sem"]].round(2).to_string(index=False))

print("\n=== F. LAS 9 POSICIONES MUTADAS (Nanobody) ===")
print(df_r[(df_r.cadena == "B") & (df_r.num.isin(MUTADOS_B))].sort_values("num")[cols_tab].round(2).to_string(index=False))

print("\n=== F. INTERFAZ NO MUTADA (Nanobody) ===")
print(df_r[(df_r.cadena == "B") & (df_r.num.isin(INTERFAZ_NO_MUT_B))].sort_values("num")[cols_tab].round(2).to_string(index=False))

print("\n=== F. RESIDUOS CLAVE DEL ANTÍGENO (RNasa A) ===")
print(df_r[(df_r.cadena == "A") & (df_r.num.isin(CLAVE_ANT_A))].sort_values("num")[cols_tab].round(2).to_string(index=False))

print("\n=== F. RESUMEN POR REGIÓN (Suma de ΔΔG_i) ===")
print(f"Mutados nanobody (9 pos):       {df_r[df_r.categoria == 'Mutado (NB)'].ddG_i.sum():+.2f} kcal/mol")
print(f"Interfaz no mutada NB (9 pos):  {df_r[df_r.categoria == 'Interfaz no mutado (NB)'].ddG_i.sum():+.2f} kcal/mol")
print(f"Resto del nanobody:             {df_r[(df_r.cadena == 'B') & (df_r.categoria == 'Otro')].ddG_i.sum():+.2f} kcal/mol")
print(f"Antígeno (RNasa A completo):    {df_r[df_r.cadena == 'A'].ddG_i.sum():+.2f} kcal/mol")

fig, ax = plt.subplots(figsize=(7.5, 5.5), layout="constrained")
top = df_r.head(18)[::-1]
nombres = [f"{c}:{a}{n}" + (f"→{b}" if a != b else "") for c, a, b, n in zip(top.cadena, top.res_2A, top.res_2B, top.num)]
colores = ["#2a9d8f" if v < 0 else "#e76f51" for v in top.ddG_i]
ax.barh(nombres, top.ddG_i, xerr=top.ddG_sem, color=colores, height=0.65, error_kw=dict(lw=1))
ax.axvline(0, color="#444", lw=1)
ax.set(xlabel="ΔΔG_i = ΔG_i(2B) − ΔG_i(2A) (kcal/mol)", title="Top 18 residuos que más cambian su contribución")
fig.savefig(f"{OUT}/descomposicion_top_ddg.png")
plt.close(fig)
print("\nArchivos guardados en:", OUT)


In [ ]:
# Asegurar que existe el directorio antes de escribir en él
!mkdir -p /content/work/resultados_energia

# Ejecutar el script utilizando el intérprete de python del entorno correcto de conda
!/usr/local/envs/gmxMMPBSA/bin/python /content/work/analisis_pareja2.py | tee /content/work/resultados_energia/resumen_energia.txt

# Mostrar las gráficas generadas de manera segura si existen
import os
from IPython.display import Image, display

for img in ("mmgbsa_componentes.png", "mmgbsa_serie_tiempo.png", "descomposicion_top_ddg.png"):
    ruta_img = f"/content/work/resultados_energia/{img}"
    if os.path.exists(ruta_img):
        display(Image(ruta_img))
    else:
        print(f"Advertencia: {img} no pudo ser encontrada en la ruta.")

In [ ]:
%%bash
# 8. Empaquetar todos los resultados para descargar y guardar en D_F_energia/
cd /content/work
rm -f resultados_energia_pareja2.zip
zip -qr resultados_energia_pareja2.zip   resultados_energia   2A_colab/mmgbsa/FINAL_* 2A_colab/mmgbsa/mmgbsa.in 2A_colab/mmgbsa/gmx_MMPBSA.log   2B_colab/mmgbsa/FINAL_* 2B_colab/mmgbsa/mmgbsa.in 2B_colab/mmgbsa/gmx_MMPBSA.log
ls -lh resultados_energia_pareja2.zip


In [ ]:
from google.colab import files
files.download("/content/work/resultados_energia_pareja2.zip")
